# 06 · Verify and register literature baselines

This notebook implements **an adapted Almazaydeh et al. (2012) linear SVM**, not author-code reproduction or a replication of its reported accuracy. [Source DOI](https://doi.org/10.1109/EMBC.2012.6347100); [screening and fidelity report](../docs/literature-baselines.md).

Run Notebook 05 first using the same `APNEA_PORTFOLIO_RUN` and smoke setting. This notebook verifies source receipts and protocol artifacts, extracts deterministic local features, and fits only a development model. It does not evaluate outer-held-out predictions or select methods from performance. The existing CNN remains unchanged; Plan 003 restores its 100 Hz input.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def root_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


RUN = root_path(
    os.environ.get("APNEA_PORTFOLIO_RUN", "data/processed/portfolio-v2/plan001-real-20261108")
)
PHASE1 = root_path(os.environ.get("APNEA_RUN", "data/processed/phase1"))
SMOKE = os.environ.get("APNEA_PORTFOLIO_SMOKE") == "1"

## 1. Freeze the method correspondence

| Source component (methods III.C–F, p.4940) | This adaptation |
| --- | --- |
| Short selected epochs | All retained project 60-second minutes |
| Source peak detector | Existing GQRS/plausibility mask; original adjacency preserved |
| Local RR statistics and directional counts | Eleven features in the fixed order below |
| Recording-wide summaries | Omitted to avoid adding longer input context |
| Linear-kernel SVM with SMO | libsvm linear `SVC`, fresh fold-local fitting |
| Source random 80/20 split | Saved grouped nested folds |
| Unspecified estimator/statistic defaults | Explicit ddof=1, linear quantiles, train median imputation/scaling, C/class weighting search |

Counts distinguish RR decreases and increases >50 ms. Each directional fraction divides by the number of accepted RR intervals, not the number of valid pairs, and is a fraction rather than a percentage. Detected beats are **not audited NN intervals**, so feature names use RR terminology.

In [ ]:
def literature_feature_names():
    return [
        "mean_rr_ms",
        "sdrr_ms",
        "rr50_decrease_count",
        "rr50_increase_count",
        "prr50_decrease_fraction",
        "prr50_increase_fraction",
        "sdsd_ms",
        "rmssd_ms",
        "median_rr_ms",
        "iqr_rr_ms",
        "mean_absolute_rr_deviation_ms",
    ]


def rr_feature_vector(seconds, accepted, min_intervals=20, min_pairs=10, adjacency=None):
    seconds = np.asarray(seconds, dtype=float)
    accepted = np.asarray(accepted, dtype=bool)
    if seconds.ndim != 1 or seconds.shape != accepted.shape or min_intervals < 2 or min_pairs < 1:
        raise ValueError("Invalid interval vector or minimum counts")
    if not np.isfinite(seconds).all() or (seconds <= 0).any():
        raise ValueError("RR intervals must be finite and positive")
    pair_mask = accepted[:-1] & accepted[1:]
    if adjacency is not None:
        adjacency = np.asarray(adjacency, dtype=bool)
        if adjacency.shape != pair_mask.shape:
            raise ValueError("Adjacency mask length mismatch")
        pair_mask &= adjacency
    selected = seconds[accepted] * 1000
    differences = np.diff(seconds * 1000)[pair_mask]
    result = np.full(11, np.nan)
    if len(selected) < min_intervals:
        return result
    mean = np.mean(selected)
    result[[0, 1, 8, 9, 10]] = [
        mean,
        np.std(selected, ddof=1),
        np.median(selected),
        np.quantile(selected, 0.75, method="linear") - np.quantile(selected, 0.25, method="linear"),
        np.mean(np.abs(selected - mean)),
    ]
    if len(differences) >= min_pairs:
        decrease, increase = np.sum(differences < -50), np.sum(differences > 50)
        result[2:6] = [decrease, increase, decrease / len(selected), increase / len(selected)]
        result[6] = np.std(differences, ddof=1) if len(differences) >= 2 else np.nan
        result[7] = np.sqrt(np.mean(differences**2))
    return result


def extract_literature_features(cohort, rr):
    if cohort.duplicated(["record_id", "minute"]).any():
        raise ValueError("Duplicate windows")
    rows = []
    for record, windows in cohort.groupby("record_id", sort=True):
        intervals = rr.loc[rr.record_id == record].sort_values("start_sample")
        starts = intervals.start_sample.to_numpy()
        ends = intervals.end_sample.to_numpy()
        seconds = intervals.rr_seconds.to_numpy()
        plausible = intervals.plausible.to_numpy(dtype=bool)
        for row in windows.itertuples():
            lo, hi = np.searchsorted(starts, [row.start_sample, row.end_sample])
            accepted = plausible[lo:hi] & (ends[lo:hi] < row.end_sample)
            adjacency = ends[lo:hi][:-1] == starts[lo:hi][1:]
            values = rr_feature_vector(seconds[lo:hi], accepted, adjacency=adjacency)
            rows.append(
                dict(
                    record_id=record,
                    minute=row.minute,
                    **dict(zip(literature_feature_names(), values, strict=True)),
                )
            )
    features = pd.DataFrame(rows)
    result = cohort.merge(features, on=["record_id", "minute"], validate="one_to_one")
    if len(result) != len(cohort):
        raise ValueError("Feature extraction changed cohort")
    return result


def fit_literature_model(features, labels, C=1.0, class_weight=None, seed=67):
    x = features.loc[:, literature_feature_names()]
    if x.isna().all().any() or set(labels) != {0, 1}:
        raise ValueError("All-missing training predictor or class-degenerate fitting labels")
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        SVC(kernel="linear", C=C, class_weight=class_weight, probability=False, random_state=seed),
    )
    return model.fit(x, labels)


def predict_literature_scores(model, features):
    scores = model.decision_function(features.loc[:, literature_feature_names()])
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite SVM decision scores")
    return scores

## 2. Simple references and score semantics

Always-negative and always-positive strategies have explicit fixed predictions. The prevalence strategy fits a single constant **only on supplied training labels**, with threshold 0.5. SVM decision scores are ranking scores, not probabilities or clinical confidence.

In [ ]:
def constant_references(train_labels, evaluation, seed=67):
    y = np.asarray(train_labels)
    if len(y) == 0 or not set(y).issubset({0, 1}):
        raise ValueError("Invalid training labels")
    keys = ["record_id", "minute", "subject_group", "label", "outer_fold"]
    if evaluation.duplicated(["record_id", "minute"]).any():
        raise ValueError("Duplicate evaluation keys")
    rows = []
    for name, score in [
        ("constant_negative", 0.0),
        ("constant_positive", 1.0),
        ("training_prevalence", float(np.mean(y))),
    ]:
        part = evaluation[keys].copy()
        part["model"], part["feature_set"], part["seed"] = name, "none", seed
        part["score"], part["score_kind"], part["threshold"] = score, "constant_probability", 0.5
        part["prediction"] = int(score >= 0.5)
        rows.append(part)
    return pd.concat(rows, ignore_index=True)


def baseline_specification(source):
    return dict(
        method_id="almazaydeh2012_local_rr_linear_svm_adapted",
        status="adapted",
        main_matrix=True,
        source_doi=source["doi"],
        source_sha256=source["sha256"],
        representation="11 local RR features",
        context_seconds=60,
        input_hz=100,
        required_signals=["single-lead ECG", "project detected RR intervals"],
        feature_order=literature_feature_names(),
        feature_definitions={
            "mean_rr_ms": "mean(r)",
            "sdrr_ms": "std(r, ddof=1)",
            "rr50_decrease_count": "sum(d < -50 ms)",
            "rr50_increase_count": "sum(d > 50 ms)",
            "prr50_decrease_fraction": "decrease_count/n",
            "prr50_increase_fraction": "increase_count/n",
            "sdsd_ms": "std(d, ddof=1)",
            "rmssd_ms": "sqrt(mean(d**2))",
            "median_rr_ms": "median(r)",
            "iqr_rr_ms": "linear_quantile(r,.75)-linear_quantile(r,.25)",
            "mean_absolute_rr_deviation_ms": "mean(abs(r-mean(r)))",
        },
        interval_policy="plausible within-minute endpoints; preserve original adjacency",
        minimum_intervals=20,
        minimum_pairs=10,
        normalization="fit-only median imputation then StandardScaler",
        architecture=None,
        reference_settings={"kernel": "linear", "C": 1, "class_weight": None},
        tuning={
            "tool": "GridSearchCV",
            "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 1000],
            "class_weight": [None, "balanced"],
            "candidates": 24,
            "score": "average_precision",
        },
        score_kind="decision_function",
        calibration=False,
        adaptations=[
            "60s instead of selected 10/15/30s epochs",
            "no recording-wide mean/SD",
            "project GQRS and quality policy",
            "explicit sample SD and quantile conventions",
            "fold-local imputation/scaling and grouped evaluation",
            "new tuning/threshold protocol",
        ],
        fidelity="independent method adaptation, not author-code reproduction",
        fixture={
            "rr_seconds": [0.8, 0.9, 1.0, 0.8],
            "accepted": [True] * 4,
            "minimum_intervals": 4,
            "minimum_pairs": 3,
        },
    )


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def verify_protocol(run, smoke):
    path = Path(run) / "protocol"
    manifest = json.loads((path / "manifest.json").read_text())
    if manifest["configuration"]["synthetic"] != smoke:
        raise ValueError("Synthetic/real protocol mismatch")
    for name, digest in manifest["artifacts"].items():
        if hash_file(path / name) != digest:
            raise ValueError("Protocol artifact hash mismatch")
    return manifest


def save_baselines(run, specification, register, features, sources):
    target = Path(run) / "baselines"
    identity = dict(
        specification=specification,
        register=register,
        sources=sources,
        versions={k: importlib.metadata.version(k) for k in ["numpy", "pandas", "scikit-learn"]},
    )
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    if target.exists():
        old = json.loads((target / "manifest.json").read_text())
        if old["identity_hash"] != digest or any(
            hash_file(target / k) != v for k, v in old["artifacts"].items()
        ):
            raise ValueError("Incompatible/corrupt baseline run; choose a fresh run root")
        return old
    with tempfile.TemporaryDirectory(prefix=".baselines-", dir=run) as temporary:
        stage = Path(temporary) / "baselines"
        stage.mkdir()
        (stage / "specifications.json").write_text(json.dumps(specification, indent=2) + "\n")
        (stage / "register.json").write_text(json.dumps(register, indent=2) + "\n")
        features.to_parquet(stage / "features.parquet", index=False)
        manifest = dict(
            identity_hash=digest,
            **identity,
            artifacts={p.name: hash_file(p) for p in stage.iterdir()},
        )
        (stage / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        stage.rename(target)
    return manifest

## 3. Verify sources and extract features on the saved cohort

The source receipt records the retrieved PDF checksum and the limited correction-metadata check. No author code or pretrained weights are copied. This is an offline workflow: cache the linked source PDF at `data/references/baselines/almazaydeh2012.pdf` before a real run. Synthetic smoke mode checks numerical method behavior without requiring that download.

In [ ]:
protocol = verify_protocol(RUN, SMOKE)
cohort = pd.read_parquet(RUN / "protocol/cohort.parquet")
assignments = pd.read_csv(RUN / "protocol/fold_assignments.csv")
source_path = ROOT / "references/baselines/almazaydeh2012/source.json"
source = json.loads(source_path.read_text())
if SMOKE:
    rr = pd.DataFrame(
        [
            dict(
                record_id=r.record_id,
                start_sample=r.start_sample + j * 85,
                end_sample=r.start_sample + (j + 1) * 85,
                rr_seconds=0.85 + 0.03 * ((j + r.label) % 3),
                plausible=True,
            )
            for r in cohort.itertuples()
            for j in range(24)
        ]
    )
else:
    if hash_file(ROOT / "data/references/baselines/almazaydeh2012.pdf") != source["sha256"]:
        raise ValueError("Primary paper checksum mismatch")
    rr_path = PHASE1 / "rr.parquet"
    rr = pd.read_parquet(rr_path)
features = extract_literature_features(cohort, rr)
specification = baseline_specification(source)
register = [
    dict(
        method_id=specification["method_id"],
        status="adapted",
        main_matrix=True,
        reason="linear SVM/local RR; explicit 60s adaptation",
    ),
    dict(
        method_id="dey2018_cnn",
        status="pending",
        main_matrix=False,
        doi="10.1007/s13534-017-0055-y",
        reason="6000-sample CNN; full implementation not verified; optional deferred",
    ),
    dict(
        method_id="wang2019_lenet",
        status="contextual_only",
        main_matrix=False,
        doi="10.7717/peerj.7731",
        reason="five-minute RR/amplitude input; supplement not copied",
    ),
    dict(
        method_id="physionet_apdet",
        status="contextual_only",
        main_matrix=False,
        url="https://physionet.org/content/apdet/1.0.0/",
        reason="five-minute NN/Hilbert context and prior database development",
    ),
    *[
        dict(method_id=name, status="eligible", main_matrix=True)
        for name in ["constant_negative", "constant_positive", "training_prevalence"]
    ],
]
sources = dict(
    protocol_identity=protocol["identity_hash"],
    synthetic=SMOKE,
    notebook=hash_file(ROOT / "notebooks/06_literature_baselines.ipynb"),
    source_receipt=hash_file(source_path),
    screening=hash_file(ROOT / "docs/literature-baselines.md"),
)
if not SMOKE:
    sources["rr"] = hash_file(rr_path)
display(pd.DataFrame(register))
print("Feature rows:", len(features))
display(features[literature_feature_names()].isna().sum().rename("missing_windows").to_frame())

## 4. Development-only fitting sanity check

Use only the saved outer-fold-0 training records excluding inner-fold-0 validation records. Fit the declared reference settings and check finite decision scores and feature ordering. This is an implementation check, not a tuned performance estimate. Full fitting, candidate ranking and outer prediction generation belong to Plan 003.

In [ ]:
fit_records = assignments.query(
    "outer_fold == 0 and outer_role == 'train' and inner_fold != 0"
).record_id
training = features[features.record_id.isin(fit_records)]
model = fit_literature_model(training, training.label.to_numpy())
scores = predict_literature_scores(model, training.iloc[:16])
assert scores.shape == (min(16, len(training)),) and np.isfinite(scores).all()
print(
    "Development fit windows:",
    len(training),
    "kernel:",
    model[-1].kernel,
    "score kind: decision_function; outer predictions not evaluated",
)
# Persist eligible artifacts only after the development fitting check succeeds.
manifest = save_baselines(RUN, specification, register, features, sources)
print("Verified baseline identity:", manifest["identity_hash"])

## Selection decision

Include the three simple references and the **adapted local-RR linear SVM** in the proposed main matrix. Do not call it a reproduction of the paper's 96.5% result. Keep Wang/apdet as contextual literature and the optional Dey model pending, with no effect on the fixed existing CNN. Candidate search and timing must still be frozen before new outer evaluation. All results remain exploratory and patient independence unverified.